# 🟡 Solution: ALiBi Attention

Reference solution for ALiBi (Attention with Linear Biases).

$$m_h = \frac{1}{2^{8h/H}}, \quad \text{bias}[h,i,j] = -m_h |i-j|$$

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def alibi_attention(Q, K, V, num_heads):
    """
    ALiBi (Attention with Linear Biases) 注意力机制。

    核心思想：不使用位置编码，而是在注意力分数上加一个线性偏置。
    偏置与距离成正比，距离越远惩罚越大（注意力越弱）。
    每个头有不同的斜率 m_h，使得不同头关注不同距离范围。

    参数:
        Q, K, V: 形状 (B, S, D) 的查询、键、值张量
            B = batch size, S = 序列长度, D = 模型维度
        num_heads: 注意力头数 H

    返回: (B, S, D) 的输出张量
    """
    B, S, D = Q.shape
    d_h = D // num_heads  # 每个头的维度

    # ---- Step 1: 计算每个头的斜率 m_h ----
    # 公式: m_h = 1 / 2^(8h/H), h = 1, 2, ..., H
    # 直觉: 第 1 个头斜率最小（关注远距离），最后一个头斜率最大（只关注近距离）
    # 例如 4 个头: m = [2^(-2), 2^(-4), 2^(-6), 2^(-8)] = [0.25, 0.0625, 0.0156, 0.0039]
    h_idx = torch.arange(1, num_heads + 1, dtype=torch.float32, device=Q.device)
    slopes = 1.0 / (2.0 ** (8.0 * h_idx / num_heads))  # shape: (H,)

    # ---- Step 2: 构建距离矩阵 |i - j| ----
    # pos = [0, 1, 2, ..., S-1], shape: (S,)
    # dist[i, j] = |i - j|, shape: (S, S)
    # 这是一个对称矩阵，对角线为 0，越远离对角线值越大
    pos = torch.arange(S, device=Q.device).float()
    dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()  # (S, S)

    # ---- Step 3: 构建偏置矩阵 ----
    # bias[h, i, j] = -m_h * |i - j|
    # 负号是因为要「惩罚」远距离，加到注意力分数上后会使远距离的分数更小
    # slopes: (H,) -> (H, 1, 1)  通过广播乘以 dist: (S, S)
    # 结果 shape: (H, S, S)
    bias = -slopes.view(num_heads, 1, 1) * dist.unsqueeze(0)

    # ---- Step 4: 标准多头注意力 ----
    # 将 Q, K, V 从 (B, S, D) reshape 为 (B, num_heads, S, d_h)
    # .view(B, S, num_heads, d_h) 把 D 维拆成 num_heads 个 d_h
    # .transpose(1, 2) 把 head 维提到第 2 维，方便后续矩阵乘法
    Qh = Q.view(B, S, num_heads, d_h).transpose(1, 2)  # (B, H, S, d_h)
    Kh = K.view(B, S, num_heads, d_h).transpose(1, 2)  # (B, H, S, d_h)
    Vh = V.view(B, S, num_heads, d_h).transpose(1, 2)  # (B, H, S, d_h)

    # ---- Step 5: 计算注意力分数 + 偏置 ----
    # Qh @ Kh^T: (B, H, S, d_h) @ (B, H, d_h, S) = (B, H, S, S)
    # 除以 sqrt(d_h) 做缩放，防止点积值过大导致 softmax 梯度消失
    # 加上 ALiBi 偏置: (H, S, S) 通过广播扩展到 (B, H, S, S)
    scores = (Qh @ Kh.transpose(-2, -1)) / (d_h ** 0.5) + bias.unsqueeze(0)

    # ---- Step 6: softmax + 加权求和 ----
    # softmax 在最后一个维度（key 维度）上归一化
    # attn @ Vh: (B, H, S, S) @ (B, H, S, d_h) = (B, H, S, d_h)
    attn = torch.softmax(scores, dim=-1)
    out = (attn @ Vh).transpose(1, 2).reshape(B, S, D)

    return out

In [ ]:
# Verify
torch.manual_seed(0)
B, S, D, H = 2, 6, 16, 4
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

out = alibi_attention(Q, K, V, num_heads=H)
print("Output shape:", out.shape)

h_idx = torch.arange(1, H + 1, dtype=torch.float32)
slopes = 1.0 / (2.0 ** (8.0 * h_idx / H))
print("Slopes for 4 heads:", slopes)

In [ ]:
from torch_judge import check
check("alibi")

## 📝 核心思路总结

1. **ALiBi 的核心创新**：用固定的线性偏置 `-m_h * |i-j|` 替代可学习的位置编码，直接加到注意力分数上，远距离 token 自然获得更低的注意力权重。
2. **几何级数斜率**：`m_h = 2^(-8h/H)` 使不同头关注不同距离范围——小斜率头看远距离，大斜率头看近距离，实现多尺度注意力。
3. **数值稳定 softmax**：`exp(x - max(x))` 是标准技巧，面试必考。减去最大值不改变 softmax 结果，但防止 exp 溢出。
4. **多头 reshape 技巧**：`(B, S, D) → (B, S, H, d_h) → (B, H, S, d_h)`，先 view 拆维度再 transpose 调整顺序，是 PyTorch 多头注意力的标准写法。